# درس 14: شبیه‌سازی مونت‌کارلو

متن کامل درس: [https://quantacademy.ir/courses/quant-math/monte-carlo-simulation/](https://quantacademy.ir/courses/quant-math/monte-carlo-simulation/)

داده‌های این دوره **شبیه‌سازی‌شده** هستند، نه قیمت واقعی بازار؛ هدف یاد دادن روش است. آموزشی است و توصیه سرمایه‌گذاری نیست.

سلول‌ها را از بالا به پایین اجرا کن (Shift+Enter).

## ۱. امید را با میانگین نمونه تقریب بزن

In [ ]:
import time
import numpy as np
import pandas as pd
from scipy.stats import norm

rng = np.random.default_rng(14)
p_true = norm.sf(2)
print(round(p_true, 5))
print("N          estimate  abs_error  std_error")
for N in (100, 10_000, 1_000_000):
    z = rng.standard_normal(N)
    est = (z > 2).mean()
    se = np.sqrt(p_true * (1 - p_true) / N)
    print(f"{N:<10} {est:<9.5f} {abs(est - p_true):<10.5f} {se:.5f}")

## ۲. کاهش واریانس: همان دقت با نمونه کمتر

In [ ]:
true = np.exp(0.5)
reps, N = 2000, 1000
plain = np.empty(reps)
anti = np.empty(reps)
cv = np.empty(reps)
for i in range(reps):
    z = rng.standard_normal(N)
    plain[i] = np.exp(z).mean()
    h = z[: N // 2]
    anti[i] = 0.5 * (np.exp(h) + np.exp(-h)).mean()
    zz = rng.standard_normal(N)
    x = np.exp(zz)
    beta = np.cov(x, zz)[0, 1] / zz.var(ddof=1)
    cv[i] = (x - beta * zz).mean()
print(round(true, 4))
print("method      mean    std")
for name, est in (("plain", plain), ("antithetic", anti), ("control", cv)):
    print(f"{name:<11} {est.mean():.4f}  {est.std():.4f}")

## ۳. شبیه‌سازی مسیر قیمت: افت ۵۰٪ در یک سال

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/quantacademyir/quant-math/main/data/btc_daily_5y.csv", parse_dates=["date"], index_col="date")
lr = np.log(df["close"]).diff().dropna().to_numpy()
m, s = lr.mean(), lr.std(ddof=1)
print(round(m, 5), round(s, 5), len(lr))

def summarize(logret):
    paths = np.exp(np.cumsum(logret, axis=1))
    peak = np.maximum.accumulate(np.maximum(paths, 1.0), axis=1)
    mdd = (paths / peak - 1).min(axis=1)
    final = paths[:, -1]
    return (mdd <= -0.5).mean(), (final < 1).mean(), np.median(final), np.percentile(final, 5)

n_paths, n_days = 10_000, 365
rng = np.random.default_rng(42)
normal = m + s * rng.standard_normal((n_paths, n_days))
boot = rng.choice(lr, size=(n_paths, n_days), replace=True)

print("method      P(maxDD>=50%)  P(end<start)  median_end  5%_end")
for name, x in (("normal", normal), ("bootstrap", boot)):
    r = summarize(x)
    print(f"{name:<11} {r[0]:<14.3f} {r[1]:<13.3f} {r[2]:<11.3f} {r[3]:.3f}")
p0 = summarize(normal)[0]
print(round(np.sqrt(p0 * (1 - p0) / n_paths), 4))

In [ ]:
for name, x in (("normal", normal), ("bootstrap", boot)):
    print(name, round((x.min(axis=1) <= np.log(0.85)).mean(), 4))
print(round((lr <= np.log(0.85)).mean(), 5), round(lr.min(), 3))

## ۴. سرعت: حلقه یا ماتریس؟

In [ ]:
Z = np.random.default_rng(1).standard_normal((2000, 365))

t0 = time.perf_counter()
loop_final = np.empty(2000)
for i in range(2000):
    x = 1.0
    for j in range(365):
        x *= np.exp(m + s * Z[i, j])
    loop_final[i] = x
t_loop = time.perf_counter() - t0

t0 = time.perf_counter()
vec_final = np.exp((m + s * Z).sum(axis=1))
t_vec = time.perf_counter() - t0
print(np.allclose(loop_final, vec_final), t_loop > t_vec)

## ۵. تکرارپذیری: بذر را ثبت کن

In [ ]:
a = np.random.default_rng(7).standard_normal(3)
b = np.random.default_rng(7).standard_normal(3)
print(np.array_equal(a, b))

streams = [np.random.default_rng(sq) for sq in np.random.SeedSequence(7).spawn(2)]
x, y = streams[0].standard_normal(10_000), streams[1].standard_normal(10_000)
print(abs(np.corrcoef(x, y)[0, 1]) < 0.05)